# 🌾 Agricultural Insurance Claim Prediction
**Binary Classification** | Logistic Regression vs XGBoost

> Kaggle Dataset: All Agriculture Data of India

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns, warnings, joblib
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix,
                             roc_curve, classification_report)
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

## 1. Load Dataset

In [ ]:
df = pd.read_csv('../data/agriculture_data.csv')
print('Shape:', df.shape)
df.head()

## 2. Data Cleaning

In [ ]:
print('Missing values:\n', df.isnull().sum())
df.dropna(inplace=True); df.drop_duplicates(inplace=True)
print('After cleaning:', df.shape)

## 3. Exploratory Data Analysis

In [ ]:
fig, axes = plt.subplots(2,3,figsize=(16,9))
fig.suptitle('EDA – Agricultural Insurance Claims', fontsize=15, fontweight='bold')

df['Insurance_Claim'].value_counts().plot(kind='pie', ax=axes[0,0],
    labels=['No Claim','Claim'], autopct='%1.1f%%', colors=['#2ecc71','#e74c3c'])
axes[0,0].set_title('Target Distribution')

df.boxplot(column='Rainfall_Deviation_Pct', by='Insurance_Claim', ax=axes[0,1])
axes[0,1].set_title('Rainfall Deviation vs Claim')

sns.histplot(data=df, x='Yield_Tonnes_Per_Ha', hue='Insurance_Claim',
             palette={0:'#2ecc71',1:'#e74c3c'}, ax=axes[0,2], bins=30, alpha=.7)
axes[0,2].set_title('Yield Distribution')

df.groupby('Crop')['Insurance_Claim'].mean().sort_values().plot(
    kind='barh', ax=axes[1,0], color='#3498db')
axes[1,0].set_title('Claim Rate by Crop')

df.groupby('Soil_Type')['Insurance_Claim'].mean().sort_values().plot(
    kind='barh', ax=axes[1,1], color='#9b59b6')
axes[1,1].set_title('Claim Rate by Soil')

num = ['Area_Hectares','Production_Tonnes','Yield_Tonnes_Per_Ha',
       'Rainfall_Deviation_Pct','Soil_Quality_Score','Insurance_Claim']
sns.heatmap(df[num].corr(), annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1,2])
axes[1,2].set_title('Correlations')

plt.tight_layout(); plt.show()

## 4. Feature Engineering

In [ ]:
df['Drought_Flag']   = (df['Rainfall_Deviation_Pct'] < -20).astype(int)
df['Flood_Flag']     = (df['Rainfall_Deviation_Pct'] >  30).astype(int)
df['Low_Yield_Flag'] = (df['Yield_Tonnes_Per_Ha'] < df['Yield_Tonnes_Per_Ha'].quantile(0.25)).astype(int)
df['Area_Log']  = np.log1p(df['Area_Hectares'])
df['Prod_Log']  = np.log1p(df['Production_Tonnes'])

le = LabelEncoder()
for c in ['State','Crop','Season','Soil_Type']:
    df[c+'_Enc'] = le.fit_transform(df[c])

FEATURES = ['Rainfall_Deviation_Pct','Soil_Quality_Score','Yield_Tonnes_Per_Ha',
            'Area_Log','Prod_Log','Temperature_C','Fertiliser_Kg_Ha','Irrigation',
            'Drought_Flag','Flood_Flag','Low_Yield_Flag',
            'Crop_Enc','Season_Enc','Soil_Type_Enc','State_Enc']
X = df[FEATURES]; y = df['Insurance_Claim']
print('Features:', len(FEATURES))

## 5. Train / Test Split & Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
print(f'Train: {X_train.shape}  Test: {X_test.shape}')

## 6. Model Training with Hyperparameter Tuning

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Logistic Regression
lr_gs = GridSearchCV(LogisticRegression(max_iter=1000),
    {'C':[0.01,0.1,1,10],'solver':['lbfgs','liblinear'],'class_weight':[None,'balanced']},
    cv=cv, scoring='recall', n_jobs=-1)
lr_gs.fit(X_train_s, y_train)
best_lr = lr_gs.best_estimator_
print('LR best:', lr_gs.best_params_)

# XGBoost
xgb_gs = GridSearchCV(XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42),
    {'n_estimators':[100,200],'max_depth':[3,5],'learning_rate':[0.05,0.1],'scale_pos_weight':[1,2]},
    cv=cv, scoring='recall', n_jobs=-1)
xgb_gs.fit(X_train, y_train)
best_xgb = xgb_gs.best_estimator_
print('XGB best:', xgb_gs.best_params_)

## 7. Model Evaluation

In [ ]:
def eval_model(name, model, Xte, yte):
    pred  = model.predict(Xte)
    proba = model.predict_proba(Xte)[:,1]
    print(f'\n── {name} ──')
    print(f'  Accuracy : {accuracy_score(yte,pred):.4f}')
    print(f'  Precision: {precision_score(yte,pred):.4f}')
    print(f'  Recall   : {recall_score(yte,pred):.4f}')
    print(f'  F1       : {f1_score(yte,pred):.4f}')
    print(f'  ROC-AUC  : {roc_auc_score(yte,proba):.4f}')
    return pred, proba

lr_pred,  lr_proba  = eval_model('Logistic Regression', best_lr,  X_test_s, y_test)
xgb_pred, xgb_proba = eval_model('XGBoost',             best_xgb, X_test,   y_test)

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(18,5))
for ax, pred, title in zip(axes[:2],[lr_pred,xgb_pred],
                            ['Logistic Regression','XGBoost']):
    sns.heatmap(confusion_matrix(y_test,pred), annot=True, fmt='d',
                cmap='Blues', ax=ax,
                xticklabels=['No Claim','Claim'],
                yticklabels=['No Claim','Claim'])
    ax.set_title(f'Confusion Matrix – {title}')

for proba, col, lbl in [(lr_proba,'#3498db','LR'),(xgb_proba,'#e74c3c','XGB')]:
    fpr,tpr,_ = roc_curve(y_test,proba)
    axes[2].plot(fpr,tpr,color=col,
                 label=f'{lbl} AUC={roc_auc_score(y_test,proba):.3f}')
axes[2].plot([0,1],[0,1],'k--')
axes[2].set_title('ROC Curve'); axes[2].legend()
plt.tight_layout(); plt.show()

## 8. Feature Importance (XGBoost)

In [ ]:
pd.Series(best_xgb.feature_importances_, index=FEATURES).sort_values().plot(
    kind='barh', figsize=(8,6), color='#e67e22', title='XGBoost Feature Importance')
plt.tight_layout(); plt.show()

## 9. Save Models

In [ ]:
import os; os.makedirs('../models', exist_ok=True)
joblib.dump(best_lr,  '../models/logistic_regression.pkl')
joblib.dump(best_xgb, '../models/xgboost_model.pkl')
joblib.dump(scaler,   '../models/scaler.pkl')
joblib.dump(FEATURES, '../models/features.pkl')
print('Models saved ✓')